# IRIS-X ARC-AGI-2 Trial 001 — offline Kaggle adapter
Run only after a separate explicit owner authorization. Attach the private unpacked-model Dataset and the private Kaggle-extracted trial Dataset at `/kaggle/input/datasets/danielwuu/trial-input`, select one NVIDIA RTX PRO 6000 96 GB, and disable Internet before starting.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
TRIAL_DATASET_ROOT = INPUT / 'datasets' / 'danielwuu' / 'trial-input'
MAX_FILES = 4097
MAX_FILE = 32 * 1024**3
MAX_TOTAL = 100 * 1024**3

def digest(data): return 'sha256:' + hashlib.sha256(data).hexdigest()
def file_digest(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024): hasher.update(chunk)
    return 'sha256:' + hasher.hexdigest()

def verify_and_copy_mounted_trial(source_root, target, expected_kind):
    if not source_root.is_dir(): raise RuntimeError(f'bound extracted trial Dataset is missing: {source_root}')
    all_paths = list(source_root.rglob('*'))
    if any(path.is_symlink() for path in all_paths): raise RuntimeError('trial Dataset contains a symbolic link')
    files = {path.relative_to(source_root).as_posix(): path for path in all_paths if path.is_file()}
    if len(files) > MAX_FILES or 'manifest.json' not in files: raise RuntimeError('trial Dataset file-count or manifest requirement failed')
    manifest_raw = files['manifest.json'].read_bytes()
    manifest = json.loads(manifest_raw)
    canonical = (json.dumps(manifest, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8')
    if manifest_raw != canonical or manifest.get('format') != 'irisx/closed-world-zip' or manifest.get('format_version') != '1.0': raise RuntimeError('non-canonical or unsupported manifest')
    if manifest.get('package_kind') != expected_kind: raise RuntimeError('package kind mismatch')
    declared = {item['path']: item for item in manifest.get('files', [])}
    if len(declared) != len(manifest.get('files', [])) or set(declared) != set(files) - {'manifest.json'}: raise RuntimeError('closed-world membership mismatch')
    target.mkdir(parents=True, exist_ok=False)
    total = 0
    for name, entry in declared.items():
        source = files[name]; size = source.stat().st_size; total += size
        if size > MAX_FILE or total > MAX_TOTAL: raise RuntimeError(f'trial Dataset byte limit exceeded: {name}')
        destination = target.joinpath(*name.split('/')); destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
        if size != entry['byte_size'] or file_digest(destination) != entry['content_digest']: raise RuntimeError(f'file binding mismatch: {name}')
    (target / 'manifest.json').write_bytes(manifest_raw)
    return manifest, digest(manifest_raw)

trial_manifest, trial_manifest_digest = verify_and_copy_mounted_trial(TRIAL_DATASET_ROOT, WORKING / 'irisx-trial', 'irisx/arc-agi-2-trial-input')
if file_digest(WORKING / 'irisx-trial/runtime/runner.py') != trial_manifest['metadata']['runner_digest']: raise RuntimeError('runner digest mismatch')
if file_digest(WORKING / 'irisx-trial/runtime/notebook.ipynb') != trial_manifest['metadata']['notebook_digest']: raise RuntimeError('notebook digest mismatch')
provenance_path = WORKING / 'irisx-trial/bindings/model-source-provenance.json'
provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
unsigned = {key: provenance[key] for key in ('schema', 'schema_version', 'artifact_id', 'version', 'payload')}
if provenance.get('schema') != 'irisx/model-source-provenance' or provenance.get('content_digest') != digest(json.dumps(unsigned, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')).encode('utf-8')): raise RuntimeError('model provenance artifact is invalid')
if provenance['content_digest'] != trial_manifest['metadata']['model']['source_provenance_digest']: raise RuntimeError('model provenance binding mismatch')
dataset = trial_manifest['metadata']['model_dataset']
model_root = INPUT / 'datasets' / dataset['owner'] / dataset['slug']
if not model_root.is_dir(): raise RuntimeError(f'bound raw model Dataset is missing: {model_root}')
os.environ['IRISX_KAGGLE_INTERNET_DISABLED'] = '1'
runner = WORKING / 'irisx-trial/runtime/runner.py'
subprocess.run([sys.executable, str(runner), '--trial-root', str(WORKING / 'irisx-trial'), '--model-root', str(model_root), '--output', str(WORKING / 'result-package.zip'), '--trial-manifest-digest', trial_manifest_digest], check=True)
print(WORKING / 'result-package.zip')
